# 🧬 Benchmarking Lightweight Mamba Models for Polyp Segmentation (1-Seed Baseline)
> **Repository:** Polyp_Segmentation_Research
> **Mô hình mục tiêu:** `ultralight_vmunet`, `lightm_unet`, `bampolyp`, `h_vmunet`
> **Cấu hình:** Seed `42`, 100 epochs, `structure_loss`, tự động profile Edge AI (Params, FLOPs, Latency, FPS, GPU Mem) và đánh giá trên 5 tập dữ liệu chuẩn.
> **Mục đích:** Chạy 1 seed trước để lấy baseline đối chiếu, tiết kiệm tài nguyên GPU phục vụ cho bước nghiên cứu cải tiến kiến trúc tiếp theo.

In [ ]:
# 1. Clone repository & cài đặt các thư viện phụ thuộc cơ bản
!git clone https://github.com/doantrongthai/Polyp_Segmentation_Research.git
%cd Polyp_Segmentation_Research
!pip install -q -r requirements.txt einops timm monai

### ⚙️ Cài đặt Mamba đã compile (Compiled Wheel)
> **Dán lệnh tải & cài đặt file wheel (.whl) Mamba của bạn vào cell trống bên dưới:**

In [ ]:
# ==============================================================================
# CELL DÀNH RIÊNG ĐỂ BẠN ĐƯA LỆNH TẢI & CÀI ĐẶT MAMBA (COMPILED WHEEL) VÀO ĐÂY
# Ví dụ:
# !pip install /path/to/causal_conv1d-*.whl
# !pip install /path/to/mamba_ssm-*.whl
# hoặc tải từ Google Drive / GitHub Releases:
# !pip install gdown
# !gdown <file_id_causal_conv1d> && pip install causal_conv1d-*.whl
# !gdown <file_id_mamba_ssm> && pip install mamba_ssm-*.whl
# ==============================================================================



In [ ]:
# 2. Kiểm tra Mamba & danh sách mô hình đã đăng ký trong framework
try:
    import mamba_ssm
    print(f"✅ mamba_ssm đã sẵn sàng!")
except Exception as e:
    print(f"⚠️ Lưu ý về mamba_ssm: {e}")

from models import list_models
print("\nDanh sách model có sẵn trong Registry:")
for m in sorted(list_models()):
    if any(k in m for k in ['mamba', 'vmunet', 'bampolyp']):
        print(f"  -> {m} (Mamba Candidate)")

---
## 🚀 Model 1: UltraLight VM-UNet (0.049M Params)
Paper: *UltraLight VM-UNet: Parallel Vision Mamba Designed for Ultra-lightweight Medical Image Segmentation* (2024)

In [ ]:
# --- 1. RUN ULTRALIGHT VM-UNET ---
!rm -rf snapshots/ results/ visualizations/ logs/

!python train.py --model ultralight_vmunet --loss structure_loss --seed 42

!python visualize.py --model ultralight_vmunet --weights snapshots/ultralight_vmunet/best.pth --dataset all --num_samples 5

!zip -q -r /content/PolypSeg_ultralight_vmunet_seed42.zip snapshots/ results/ visualizations/ logs/
from google.colab import files
files.download('/content/PolypSeg_ultralight_vmunet_seed42.zip')

---
## 🚀 Model 2: LightM-UNet (1.34M Params)
Paper: *LightM-UNet: Mamba Assists in Lightweight UNet for Medical Image Segmentation* (2024)

In [ ]:
# --- 2. RUN LIGHTM-UNET ---
!rm -rf snapshots/ results/ visualizations/ logs/

!python train.py --model lightm_unet --loss structure_loss --seed 42

!python visualize.py --model lightm_unet --weights snapshots/lightm_unet/best.pth --dataset all --num_samples 5

!zip -q -r /content/PolypSeg_lightm_unet_seed42.zip snapshots/ results/ visualizations/ logs/
from google.colab import files
files.download('/content/PolypSeg_lightm_unet_seed42.zip')

---
## 🚀 Model 3: BAMPolyp (6.66M Params)
Paper: *BAMPolyp: Bi-Axial Mamba Bottleneck for Gastrointestinal Polyp Segmentation* (ICCV 2025 Workshop / CVAMD)

In [ ]:
# --- 3. RUN BAMPOLYP ---
!rm -rf snapshots/ results/ visualizations/ logs/

!python train.py --model bampolyp --loss structure_loss --seed 42

!python visualize.py --model bampolyp --weights snapshots/bampolyp/best.pth --dataset all --num_samples 5

!zip -q -r /content/PolypSeg_bampolyp_seed42.zip snapshots/ results/ visualizations/ logs/
from google.colab import files
files.download('/content/PolypSeg_bampolyp_seed42.zip')

---
## 🚀 Model 4: H-vmunet (8.97M Params)
Paper: *H-vmunet: High-order Vision Mamba UNet for Medical Image Segmentation* (Neurocomputing 2025)

In [ ]:
# --- 4. RUN H-VMUNET ---
!rm -rf snapshots/ results/ visualizations/ logs/

!python train.py --model h_vmunet --loss structure_loss --seed 42

!python visualize.py --model h_vmunet --weights snapshots/h_vmunet/best.pth --dataset all --num_samples 5

!zip -q -r /content/PolypSeg_h_vmunet_seed42.zip snapshots/ results/ visualizations/ logs/
from google.colab import files
files.download('/content/PolypSeg_h_vmunet_seed42.zip')

---
## 🔄 Tùy Chọn: Chạy Tự Động Lần Lượt Cả 4 Model (All-in-One Sequential Runner)
> Cell này sẽ tự động chạy tuần tự từng model một, tạo zip độc lập và tải xuống riêng biệt từng file.

In [ ]:
import os
import shutil
from google.colab import files

mamba_models = ['ultralight_vmunet', 'lightm_unet', 'bampolyp', 'h_vmunet']

for model_name in mamba_models:
    print(f"\n=======================================================")
    print(f"BẮT ĐẦU TRAINING & BENCHMARK: {model_name} (Seed 42)")
    print(f"=======================================================\n")
    
    # 1. Dọn dẹp thư mục kết quả để zip không bị lẫn lộn giữa các model
    for d in ['snapshots', 'results', 'visualizations', 'logs']:
        if os.path.exists(d):
            shutil.rmtree(d)
            
    # 2. Huấn luyện 1 seed duy nhất với structure_loss
    !python train.py --model {model_name} --loss structure_loss --seed 42
    
    # 3. Trực quan hóa mask dự đoán
    weights_path = f"snapshots/{model_name}/best.pth"
    if os.path.exists(weights_path):
        !python visualize.py --model {model_name} --weights {weights_path} --dataset all --num_samples 5
    
    # 4. Đóng gói zip riêng cho model này
    zip_filename = f"/content/PolypSeg_{model_name}_seed42.zip"
    !zip -q -r {zip_filename} snapshots/ results/ visualizations/ logs/
    print(f"✅ Đã tạo file zip riêng cho {model_name}: {zip_filename}")
    
    # 5. Tự động tải về máy
    try:
        files.download(zip_filename)
    except Exception as e:
        print(f"Lưu ý tải browser: {e}. File zip đã lưu tại {zip_filename}")

print("\n🎉 HOÀN TẤT HUẤN LUYỆN TOÀN BỘ 4 MÔ HÌNH MAMBA!")